In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('/content/customer_churn_sample (2).csv')

# Display the first five records
print(df.head())

# Display the number of rows and columns
print("Dataset shape:", df.shape)

# Display all column names
print("Columns:", df.columns.tolist())

In [ ]:
# Check data types and non-null values
df.info()

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check duplicate customer IDs
print("Duplicate customer IDs:", df["CustomerID"].duplicated().sum())

# Check numerical statistics
print("\nNumerical summary:")
print(df.describe())

In [ ]:
# Check whether important numeric fields contain invalid values
print("Invalid ages:", (~df["Age"].between(0, 120)).sum())

print(
    "Invalid tenure:",
    (df["TenureMonths"] < 0).sum()
)

print(
    "Invalid monthly charges:",
    (df["MonthlyCharges"] < 0).sum()
)

print(
    "Invalid support tickets:",
    (df["SupportTickets"] < 0).sum()
)

# Check allowed churn values
print("\nChurn categories:")
print(df["Churn"].value_counts(dropna=False))

# Check contract categories
print("\nContract categories:")
print(df["ContractType"].value_counts(dropna=False))

Feature Selection: TenureMonths, MonthlyCharges, SupportTickets, ContractType, and SubscriptionType were selected because they describe customer tenure, spending, support needs, and account characteristics. Churn was retained as the observed outcome for evaluating the customer segments. CustomerID was excluded from the analysis because it is a unique identifier.

In [ ]:
def assign_segment(tenure):
    if pd.isna(tenure):
        return "Unknown"
    elif tenure < 12:
        return "New Customers"
    elif tenure <= 24:
        return "Established Customers"
    else:
        return "Long-Term Customers"


df["CustomerSegment"] = df["TenureMonths"].apply(assign_segment)

print(
    df[
        ["CustomerID", "TenureMonths", "CustomerSegment", "Churn"]
    ].to_string(index=False)
)

In [ ]:
segment_summary = (
    df.groupby("CustomerSegment")
      .agg(
          CustomerCount=("CustomerID", "count"),
          AverageTenure=("TenureMonths", "mean"),
          AverageMonthlyCharges=("MonthlyCharges", "mean"),
          AverageSupportTickets=("SupportTickets", "mean")
      )
      .round(2)
)

print(segment_summary)

In [ ]:
segment_summary["CustomerCount"].plot(
    kind="bar",
    title="Number of Customers by Segment",
    xlabel="Customer Segment",
    ylabel="Number of Customers",
    rot=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Convert churn labels into a consistent format
df["Churn"] = df["Churn"].astype("string").str.strip().str.title()

# Check for unexpected values
print(df["Churn"].value_counts(dropna=False))

# Calculate churn rates by segment
churn_summary = (
    df.groupby("CustomerSegment")
      .agg(
          TotalCustomers=("CustomerID", "count"),
          ChurnedCustomers=("Churn", lambda x: x.eq("Yes").sum()),
          ObservedChurnRate=("Churn", lambda x: x.eq("Yes").mean() * 100)
      )
      .round(2)
)

print(churn_summary)

In [ ]:
def calculate_churn_risk(row):
    score = 0

    # Rule 1: New customers
    if row["TenureMonths"] < 12:
        score += 30

    # Rule 2: High support-ticket count
    if row["SupportTickets"] >= 4:
        score += 40

    # Rule 3: Month-to-month contract
    if row["ContractType"] == "Month-to-Month":
        score += 30

    return score


df["ChurnRiskScore"] = df.apply(calculate_churn_risk, axis=1)

In [ ]:
def classify_risk(score):
    if pd.isna(score):
        return "Unknown"
    elif score < 30:
        return "Low Risk"
    elif score < 60:
        return "Medium Risk"
    else:
        return "High Risk"


df["RiskCategory"] = df["ChurnRiskScore"].apply(classify_risk)

print(
    df[
        [
            "CustomerID",
            "CustomerSegment",
            "TenureMonths",
            "SupportTickets",
            "ContractType",
            "ChurnRiskScore",
            "RiskCategory",
            "Churn"
        ]
    ].to_string(index=False)
)

In [ ]:
# Test the segment boundaries
assert assign_segment(11) == "New Customers"
assert assign_segment(12) == "Established Customers"
assert assign_segment(24) == "Established Customers"
assert assign_segment(25) == "Long-Term Customers"

# Test the risk-category boundaries
assert classify_risk(0) == "Low Risk"
assert classify_risk(29) == "Low Risk"
assert classify_risk(30) == "Medium Risk"
assert classify_risk(59) == "Medium Risk"
assert classify_risk(60) == "High Risk"
assert classify_risk(100) == "High Risk"

# Test the maximum score
test_customer = pd.Series({
    "TenureMonths": 6,
    "SupportTickets": 5,
    "ContractType": "Month-to-Month"
})

assert calculate_churn_risk(test_customer) == 100

print("All boundary and scoring tests passed!")

In [ ]:
final_table = df[
    [
        "CustomerID",
        "CustomerSegment",
        "TenureMonths",
        "SubscriptionType",
        "MonthlyCharges",
        "SupportTickets",
        "ContractType",
        "Churn",
        "ChurnRiskScore",
        "RiskCategory"
    ]
].copy()

print(final_table.to_string(index=False))

In [ ]:
final_table.to_csv(
    "customer_segmentation_results.csv",
    index=False
)

print("Final segment table saved successfully.")

# Customer Churn Segmentation and Recommendation Model

## 1. Project Objective

The objective of this project is to analyze customer behavior, divide customers into meaningful segments, and create a transparent rule-based churn-risk score. The analysis helps identify customer groups that may benefit from retention support.

## 2. Dataset Overview

The project uses a customer churn dataset containing 15 customer records.

The dataset includes customer information such as tenure, monthly charges, support tickets, contract type, subscription type, and churn status.

The analysis uses Python and Pandas for data processing, segmentation, and summary calculations, along with Matplotlib for visualization.

## 3. Data Quality Assessment

Data quality checks were designed to identify missing values, duplicate records, invalid values, and unexpected categories.

The checks include:

* Missing-value analysis.
* Duplicate-row and duplicate-customer-ID checks.
* Validation of customer age, tenure, monthly charges, and support-ticket counts.
* Verification of churn and contract categories.

Any data-quality issues should be documented, along with the cleaning decisions taken. Records should not be removed without a clear reason.

## 4. Feature Selection

The following features were selected for customer segmentation and analysis:

* **TenureMonths:** Represents the number of months since signup.
* **MonthlyCharges:** Represents the customer's monthly subscription charges.
* **SupportTickets:** Indicates the number of support tickets.
* **ContractType:** Describes the customer's contract arrangement.
* **SubscriptionType:** Identifies the customer's subscription category.
* **Churn:** Records whether the customer left during the observation period.

CustomerID was excluded as a predictive feature because it is an identifier rather than a meaningful measure of customer behavior.

## 5. Customer Segmentation

Customers were divided into three groups using their tenure.

* **New Customers:** Less than 12 months of tenure.
* **Established Customers:** 12 to 24 months of tenure.
* **Long-Term Customers:** More than 24 months of tenure.

The supplied customer results show the following distribution:

| Customer Segment      | Total Customers | Churned Customers | Observed Churn Rate |
| --------------------- | --------------: | ----------------: | ------------------: |
| New Customers         |               5 |                 5 |                100% |
| Established Customers |               6 |                 2 |              33.33% |
| Long-Term Customers   |               4 |                 0 |                  0% |
| Total                 |              15 |                 7 |              46.67% |

The New Customers segment has the highest observed churn rate in this sample. The Long-Term Customers segment has no recorded churn in the supplied data.

These findings describe the sample and should not be treated as reliable estimates for the entire customer population.

## 6. Churn Analysis

The overall observed churn rate is 46.67%, calculated from 7 churned customers out of 15 records.

The New Customers segment has an observed churn rate of 100%. This makes early customer experience an important area for further investigation.

The Established Customers segment has an observed churn rate of 33.33%. Reviewing customer feedback and unresolved support issues may help the business understand the reasons for churn.

The Long-Term Customers segment has an observed churn rate of 0%. However, this result is based on only four records and does not prove that these customers have no future churn risk.

## 7. Rule-Based Churn-Risk Score

A transparent scoring system was developed using three rules.

| Condition                       | Points |
| ------------------------------- | -----: |
| Tenure is less than 12 months   |     30 |
| Support tickets are 4 or more   |     40 |
| Contract type is Month-to-Month |     30 |
| Maximum score                   |    100 |

The points are added when a customer meets the corresponding conditions.

Customers are classified into three risk categories:

* **Low Risk:** Score from 0 to 29.
* **Medium Risk:** Score from 30 to 59.
* **High Risk:** Score from 60 to 100.

This scoring system is reproducible and easy to explain. However, it is a manually defined prioritization method rather than a statistically validated prediction of future churn.

## 8. Validation and Edge-Case Testing

The segmentation and scoring functions were tested using boundary values.

The tests cover:

* Tenure values of 11, 12, 24, and 25 months.
* Risk scores of 0, 29, 30, 59, 60, and 100.
* A customer who meets all three scoring conditions.

The tests are intended to verify that customers are assigned to the correct segments and risk categories.

Missing or invalid inputs should also be handled explicitly before the scoring method is used on new customer data.

## 9. Customer Retention Recommendations

### New Customers

**Recommendation:** Prioritize reviewing onboarding and early customer experience.

The segment has an observed churn rate of 100% in the supplied sample. Investigate customer feedback, support history, and possible difficulties during onboarding.

### Established Customers

**Recommendation:** Review unresolved support issues and ask customers about their experience.

The segment has an observed churn rate of 33.33%. Customer contact should focus on understanding individual needs rather than assuming that every customer is dissatisfied.

### Long-Term Customers

**Recommendation:** Monitor customer satisfaction and review whether existing subscription plans continue to meet customer needs.

No churn was recorded in this segment during the observation period. Continue routine retention activities rather than assuming that future churn risk is zero.

### High-Risk Customers

Customers with high rule-based scores may be prioritized for supportive contact and review of unresolved service issues.

Customers already marked as churned should be handled separately, through exit feedback or an appropriate win-back process.

## 10. Limitations

This project has several limitations:

1. The dataset contains only 15 customer records.
2. The small sample makes observed churn rates highly sensitive to individual records.
3. The scoring weights were defined using simple rules and have not been statistically validated.
4. The observed relationship between customer characteristics and churn does not establish causation.
5. The recommendations have not been tested to determine whether they reduce churn.

Therefore, the results should be interpreted as an exploratory analysis rather than a production-ready churn-prediction system.

## 11. Conclusion and Next Steps

This project demonstrates how customer data can be used to create explainable segments, summarize observed churn, and prioritize customers for further investigation.

The analysis identifies New Customers as the first group to investigate because all five customers in this segment are marked as churned in the supplied sample.

The next steps are to collect more customer data, investigate the reasons for churn, contact appropriate customers, and measure retention outcomes after any intervention.

Future work could compare the rule-based score with a properly validated statistical or machine-learning model when sufficient data is available.
